# Module 01 live coding: research data workflow

Goal: start from a paper idea, inspect small data inventories, and export a paper-ready planning figure.

This notebook is intentionally small. A beginner should be able to run it top to bottom before learning advanced cleaning or plotting.

## 1. Load the tools
We use pandas for tables and matplotlib for a final static figure.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

BASE = Path.cwd()
RAW = BASE / "data" / "raw"
OUT_TABLES = BASE / "outputs" / "tables"
OUT_FIGURES = BASE / "outputs" / "figures"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

print(BASE)

## 2. Read three small CSV files
Each file answers a different beginner question: what projects exist, what sources are credible, and what research track could reuse the workflow?

In [ ]:
projects = pd.read_csv(RAW / "module01_toy_project_inventory.csv")
sources = pd.read_csv(RAW / "module01_public_source_inventory.csv")
questions = pd.read_csv(RAW / "module01_transfer_question_bank.csv")

print(projects.shape, sources.shape, questions.shape)
projects.head(3)

## 3. Ask the paper-first questions
Before plotting, write down what one row means and which claim the figure could support.

In [ ]:
inventory_view = projects[[
    "project_id", "paper_area", "unit_of_observation",
    "data_role", "figure_candidate", "readiness_note"
]]
inventory_view

## 4. Create a source readiness score
This score is not a formal metric. It is a teaching heuristic: trusted sources are good, but very complex sources need guidance.

In [ ]:
sources["readiness_score"] = sources["trust_score"] * 20 - sources["beginner_complexity"] * 8
sources["readiness_band"] = pd.cut(
    sources["readiness_score"],
    bins=[-1, 35, 55, 100],
    labels=["needs preparation", "good with guidance", "good first source"]
)

summary = sources[[
    "source_id", "source_name", "research_track", "unit_of_observation",
    "trust_score", "beginner_complexity", "readiness_score",
    "readiness_band", "figure_candidate", "paper_risk"
]].sort_values("readiness_score", ascending=False)
summary

## 5. Export a table and a figure
A paper workflow should leave output files behind, not only screen output inside the notebook.

In [ ]:
summary.to_csv(OUT_TABLES / "module01_source_readiness_summary.csv", index=False)
questions.to_csv(OUT_TABLES / "module01_paper_question_map.csv", index=False)

ordered = summary.sort_values("readiness_score")
color_map = {
    "good first source": "#16815f",
    "good with guidance": "#245ee8",
    "needs preparation": "#bd3f68",
}
colors = [color_map[str(value)] for value in ordered["readiness_band"]]

fig, ax = plt.subplots(figsize=(9.5, 5.6))
ax.barh(ordered["source_id"], ordered["readiness_score"], color=colors, height=0.64)
ax.set_xlabel("Readiness score = trust score x 20 - beginner complexity x 8")
ax.set_ylabel("Candidate data source")
ax.set_title("Which data sources are ready for a first research figure?", loc="left", weight="bold")
ax.set_xlim(0, 90)
ax.grid(axis="x", color="#e6edf7")
ax.set_axisbelow(True)
for i, (_, row) in enumerate(ordered.iterrows()):
    ax.text(row["readiness_score"] + 1.2, i, int(row["readiness_score"]), va="center", weight="bold")

for ext in ["png", "svg", "pdf"]:
    fig.savefig(OUT_FIGURES / f"module01_source_readiness.{ext}", dpi=220, bbox_inches="tight")

plt.show()

## 6. Draft a caption
The caption names the evidence, explains the score, and states a limitation.

In [ ]:
caption = (
    "Figure 1. Source readiness for a first Python research figure. "
    "The score combines source trust and beginner complexity for six candidate data sources. "
    "Higher values indicate sources that are easier to inspect in a first notebook while still supporting an academic paper claim. "
    "The score is a teaching heuristic, not a formal data-quality metric; final source choice still requires checking definitions, missingness, ethics, and citation requirements."
)

(OUT_TABLES / "module01_caption_draft.md").write_text(caption, encoding="utf-8")
print(caption)